In [ ]:
# DINOv3 ViT-7B features -- the only lever sized for 0.95+
#
# Measured progression on this task:
#   5120 -> 0.86912 | 7680 -> 0.91610 (+0.047) | 9728 -> 0.92617 (+0.010)
#   ConvNeXt-L swap -> 0.92281 (-0.003)
# Block stacking is saturating, and the host's scorer OOMs above ~10,240 dims
# (20000 x 12800 float64 = 2.05 GB), so additions are out -- only SWAPS remain.
#
# What is left is raw capacity. ViT-L -> ViT-g was worth +0.037 here; 7B is a
# bigger jump again (6.7B vs H+'s 0.8B). Its hidden size is ~4096, so CLS+GeM
# gives an ~8192-dim block -- which nearly fills the dim budget alone:
#     7B@512 (8192) + rotation-corrected ViT-L@518 (2048) = 10240 exactly.
#
# Attach: hark99/facebookdinov3-vit7b16-pretrain-lvd1689m  (no HF token needed)
import glob, os, time, gc
import numpy as np, torch
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None
ROT    = 180     # rotate before embedding
SIZE   = 512      # 512/16 = 32 -> 1024 tokens. 768 would be ~2.3x slower and
                  # this run is already the long pole.
BATCH  = 2
WORK   = Path('/kaggle/working')
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEV == 'cuda', 'a 7B on CPU is not finishable -- attach a GPU'
print('device:', torch.cuda.get_device_name(0))
CKPT, DONE = WORK / 'd7b180_ckpt.npy', WORK / 'd7b180_done.npy'

# Only the 3,439 flip CANDIDATES need a 180-degree view -- rotating a clean
# gallery scan can only hurt, and 17% of the corpus is ~45 min instead of 4 h.
# CAND indices come from the same deskew bounding-box test used before; embedded
# here so this notebook needs no extra Dataset.
import base64, zlib
_C = sorted(glob.glob('/kaggle/input/**/rot_candidates.npy', recursive=True))
assert _C, 'attach rot_candidates.npy as a Dataset'
CAND = np.load(_C[0])
print(f'{len(CAND)} candidates to embed at {ROT} degrees')

DATA = Path('/kaggle/input/competitions/lost-in-the-museum-f1/archive/kaggle_dataset/kaggle_dataset')
if not DATA.exists():
    hits = [d for d in glob.glob('/kaggle/input/**/', recursive=True)
            if glob.glob(os.path.join(d, '*.png'))]
    assert hits, 'no PNG directory under /kaggle/input'
    DATA = Path(max(hits, key=lambda h: len(glob.glob(os.path.join(h, '*.png')))))
paths = sorted(DATA.glob('*.png'))
print(len(paths), 'images from', DATA)
assert len(paths) == 20000, f'expected 20000, got {len(paths)}' 

In [ ]:
from transformers import AutoModel

# NO QUANTISATION. bitsandbytes is absent from the Kaggle image, and installing
# it mid-session does not help: transformers caches is_bitsandbytes_available()
# at import time, so it still reports missing until the kernel restarts. Two
# cycles were lost to that -- load fp16 instead and drop the dependency.
HAVE_BNB = False

local = [d for d in glob.glob('/kaggle/input/**/', recursive=True)
         if os.path.exists(os.path.join(d, 'config.json'))]
print('model dirs:'); [print('   ', d) for d in local]
pick = sorted(d for d in local if '7b' in d.lower()) or \
       sorted(d for d in local if 'dinov3' in d.lower() and 'convnext' not in d.lower())
assert pick, 'attach the DINOv3 ViT-7B mirror'
MODEL = pick[0].rstrip('/'); print('using:', MODEL)

# 7B is 13.4 GB in fp16 against a T4's 16 GB -- that leaves almost nothing for
# activations. NF4 brings it to ~3.4 GB. Quantisation does perturb the
# embeddings, but a 4-bit 7B still carries far more capacity than an fp16 0.8B,
# and there is no time to be precious about it.
if HAVE_BNB:   # never taken -- kept so a bnb-enabled image would use it
    from transformers import BitsAndBytesConfig
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.float16,
                           bnb_4bit_use_double_quant=True)
    model = AutoModel.from_pretrained(MODEL, device_map='auto',
                                      quantization_config=q).eval()
else:
    BATCH = 1          # 7B fp16 is ~13.4 GB against a T4's ~15.4 GB usable;
                       # device_map='auto' offloads the remainder to CPU, so it
                       # runs either way, just slower if it spills.
    model = AutoModel.from_pretrained(MODEL, device_map='auto',
                                      torch_dtype=torch.float16,
                                      low_cpu_mem_usage=True).eval()
assert getattr(model.config, 'model_type', '') != 'convnext', 'that is a ConvNeXt'
PATCH  = getattr(model.config, 'patch_size', 16)
HID    = getattr(model.config, 'hidden_size', 4096)
NPATCH = (SIZE // PATCH) ** 2
print(f'patch={PATCH} hidden={HID} tokens={NPATCH} -> block width {2*HID}')
print(f'GPU mem after load: {torch.cuda.memory_allocated()/2**30:.1f} GB')

In [ ]:
tf = transforms.Compose([
    transforms.Lambda(lambda im: im.rotate(ROT, expand=True)),
    transforms.Resize((SIZE, SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DS(Dataset):
    def __len__(self): return len(paths)
    def __getitem__(self, i):
        try: return tf(Image.open(paths[i]).convert('RGB')), i
        except Exception: return torch.zeros(3, SIZE, SIZE), i

@torch.no_grad()
def embed(x):
    # DINOv3 uses REGISTER tokens, so patch tokens are NOT h[:, 1:]. Taking the
    # LAST NPATCH is correct for any number of leading tokens.
    h = model(pixel_values=x).last_hidden_state
    cls = h[:, 0].float()
    gem = h[:, -NPATCH:].float().clamp(min=1e-6).pow(3.0).mean(1).pow(1/3.0)
    return torch.cat([cls, gem], 1)

@torch.no_grad()
def embed_safe(x):
    try:
        return embed(x)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(x) == 1: raise
        h = len(x) // 2
        return torch.cat([embed_safe(x[:h]), embed_safe(x[h:])])

feats = np.load(CKPT) if CKPT.exists() else np.zeros((len(paths), 2*HID), np.float32)
done  = np.load(DONE) if DONE.exists() else np.zeros(len(paths), bool)
for pat in ('d7b_ckpt.npy', 'd7b_done.npy'):     # resume from an attached partial run
    hit = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
    if hit and not CKPT.exists():
        a = np.load(hit[0])
        if pat.endswith('ckpt.npy') and a.shape == feats.shape: feats = a
        if pat.endswith('done.npy')  and a.shape == done.shape:  done = a
print(f'resuming with {done.sum()}/{len(paths)} embedded', flush=True)

todo = np.array([i for i in CAND if not done[i]], dtype=np.int64)
dl = DataLoader(DS(), batch_size=BATCH, num_workers=2, sampler=todo.tolist())
t0 = time.time(); n = 0
for x, idx in dl:
    v = embed_safe(x.to(DEV).half()).cpu().numpy()
    if n == 0:
        assert np.isfinite(v).all(), 'first batch non-finite -- 4-bit overflow'
        print(f'first batch OK, block width {v.shape[1]}', flush=True)
    feats[idx.numpy()] = v; done[idx.numpy()] = True
    n += len(idx)
    if n % (BATCH * 50) == 0:
        r = n / (time.time() - t0)
        print(f'  {done.sum()}/{len(paths)}  {r:.1f} img/s  '
              f'ETA {(len(todo)-n)/r/60:.0f} min', flush=True)
        np.save(CKPT, feats); np.save(DONE, done)
        gc.collect(); torch.cuda.empty_cache()

np.save(CKPT, feats); np.save(DONE, done)
assert np.isfinite(feats[CAND]).all(), 'non-finite on candidates -- do not use'
np.save(WORK / f'features_d7b{SIZE}_rot{ROT}.npy', feats)
print(f'wrote features_d7b{SIZE}_rot{ROT}.npy {feats.shape} in {(time.time()-t0)/60:.1f} min')
print('embedded rows (candidates only):', int(done.sum()))